# P1 — Tải PADS + Monipar và audit
Output của notebook này (`data/raw`, `reports/`) là đầu vào cho các phiên sau qua `kernel_sources`.

In [ ]:
%cd /kaggle/working
!mkdir -p scripts reports data/raw
!pip -q install tqdm

In [ ]:
%%writefile scripts/01_download.sh
#!/usr/bin/env bash
# Phiên 1: tải PADS (PhysioNet) và Monipar (Zenodo 8104853). Chạy lại được, tự nối tiếp.
set -euo pipefail
ROOT="$(cd "$(dirname "$0")/.." && pwd)"
mkdir -p "$ROOT/data/raw/pads" "$ROOT/data/raw/monipar" "$ROOT/reports"

echo "== PADS =="
cd "$ROOT/data/raw/pads"
if ! ls */file_list.csv >/dev/null 2>&1 && ! ls file_list.csv >/dev/null 2>&1; then
  curl -sSL --retry 5 -o pads.zip https://physionet.org/content/parkinsons-disease-smartwatch/get-zip/1.0.0/
  ls -la pads.zip
  unzip -q -o pads.zip && rm -f pads.zip
  # đưa nội dung thư mục con duy nhất lên gốc data/raw/pads
  d=$(ls -d */ | head -1); if [ "$(ls -d */ | wc -l)" = 1 ] && [ ! -e movement ]; then mv "$d"* . && rmdir "$d"; fi
fi
ls | head -20
# kiểm tra tổng SHA256 nếu PhysioNet cung cấp
if [ -f SHA256SUMS.txt ]; then
  sha256sum -c SHA256SUMS.txt --quiet > "$ROOT/reports/pads_sha256.log" 2>&1 \
    && echo "PADS checksum OK" || echo "PADS checksum CÓ LỖI, xem reports/pads_sha256.log"
fi

echo "== Monipar =="
cd "$ROOT"
python3 - <<'PY'
import hashlib, json, pathlib, requests
from tqdm import tqdm
out = pathlib.Path("data/raw/monipar"); out.mkdir(parents=True, exist_ok=True)
rec = requests.get("https://zenodo.org/api/records/8104853", timeout=60).json()
(out / "_zenodo_record.json").write_text(json.dumps(rec, indent=2))
for f in rec["files"]:
    name, url = f["key"], f["links"]["self"]
    dst = out / name
    want = f.get("checksum", "")  # dạng "md5:..."
    if dst.exists() and want.startswith("md5:"):
        if hashlib.md5(dst.read_bytes()).hexdigest() == want[4:]:
            print("đã có:", name); continue
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dst, "wb") as fh, tqdm(total=f.get("size"), unit="B", unit_scale=True, desc=name) as bar:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk); bar.update(len(chunk))
    if want.startswith("md5:"):
        ok = hashlib.md5(dst.read_bytes()).hexdigest() == want[4:]
        print(name, "md5 OK" if ok else "md5 SAI")
PY

# giải nén nếu là zip
cd "$ROOT/data/raw/monipar"
for z in *.zip; do [ -e "$z" ] && unzip -n -q "$z" -d "${z%.zip}"; done || true
echo "Xong. Chạy tiếp: python scripts/02_audit.py"


In [ ]:
%%writefile scripts/02_audit.py
"""Phiên 1: audit PADS và Monipar. Không giả định cứng cấu trúc thư mục: tự dò, ghi lại những gì thấy.

Kết quả: reports/audit_pads.md, reports/audit_monipar.md
"""
import collections, json, pathlib, re, html
import numpy as np
import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
REP = ROOT / "reports"; REP.mkdir(exist_ok=True)


def inventory(base: pathlib.Path, md: list):
    files = [p for p in base.rglob("*") if p.is_file()]
    by_ext = collections.Counter(p.suffix.lower() or "(none)" for p in files)
    size = sum(p.stat().st_size for p in files)
    md += [f"- Tổng file: {len(files)}, dung lượng {size/1e6:.1f} MB",
           "- Theo phần mở rộng: " + ", ".join(f"`{k}`: {v}" for k, v in by_ext.most_common())]
    dirs = collections.Counter(str(p.parent.relative_to(base)) for p in files)
    md.append("- Thư mục (top 15 theo số file):")
    md += [f"  - `{d}`: {n}" for d, n in dirs.most_common(15)]
    return files


def read_numeric(path: pathlib.Path):
    """Đọc file bảng số, tự đoán dấu phân cách và header."""
    for sep in [",", ";", "\t", r"\s+"]:
        try:
            df = pd.read_csv(path, sep=sep, engine="python", nrows=200000)
            if df.shape[1] >= 2:
                if not np.issubdtype(df.dtypes.iloc[0], np.number):
                    pass
                return df, sep
        except Exception:
            continue
    return None, None


def fs_from_time(t: np.ndarray):
    d = np.diff(t[np.isfinite(t)])
    if len(d) == 0 or np.mean(d > 0) < 0.95:  # không đơn điệu tăng → không phải cột thời gian
        return None
    d = d[d > 0]
    if len(d) == 0:
        return None
    med = np.median(d)
    # đơn vị: s, ms hoặc ns
    for scale, unit in [(1, "s"), (1e-3, "ms"), (1e-9, "ns")]:
        fs = 1 / (med * scale)
        if 5 <= fs <= 2000:
            return fs, unit, float(np.std(d) / med)
    return None


# ---------------------------------------------------------------- PADS
def audit_pads():
    base = RAW / "pads"
    md = ["# Audit PADS", ""]
    if not base.exists():
        md.append("**Chưa có dữ liệu** — chạy scripts/01_download.sh"); return md
    md.append("## Kho file"); inventory(base, md)

    # bệnh nhân
    pj = sorted(base.rglob("patient_*.json"))
    md += ["", f"## Bệnh nhân ({len(pj)} file JSON)"]
    if pj:
        rows = [json.loads(p.read_text()) for p in pj]
        pdf = pd.json_normalize(rows)
        md.append(f"- Cột: {', '.join(map(str, pdf.columns))}")
        for c in ["condition", "gender", "handedness"]:
            if c in pdf:
                md.append(f"- `{c}`: " + ", ".join(f"{k}={v}" for k, v in pdf[c].value_counts(dropna=False).items()))
        for c in ["age", "age_at_diagnosis", "height", "weight"]:
            if c in pdf:
                s = pd.to_numeric(pdf[c], errors="coerce")
                md.append(f"- `{c}`: mean {s.mean():.1f} ± {s.std():.1f}, thiếu {s.isna().sum()}")
        pdf.to_csv(REP / "pads_patients.csv", index=False)
        md.append("- Đã ghi `reports/pads_patients.csv`")

    # file_list.csv của bản preprocessed (nếu có)
    for fl in base.rglob("file_list.csv"):
        df = pd.read_csv(fl)
        md += ["", f"## `{fl.relative_to(base)}`", f"- shape {df.shape}; cột: {', '.join(df.columns)}",
               "```", df.head(5).to_string(), "```"]
        for c in df.columns:
            if df[c].nunique() <= 10:
                md.append(f"- `{c}`: " + ", ".join(f"{k}={v}" for k, v in df[c].value_counts().items()))

    # chuỗi thời gian: tên dạng {id}_{Task}_{Wrist}.txt
    ts = [p for p in base.rglob("*.txt") if re.match(r"^\d+_.+_(Left|Right)Wrist", p.stem)]
    md += ["", f"## Chuỗi thời gian thô ({len(ts)} file)"]
    if ts:
        meta = pd.DataFrame([dict(zip(["subject", "task", "wrist"], p.stem.split("_", 2)), path=str(p)) for p in ts])
        meta["subject"] = meta["subject"].astype(int)
        md.append(f"- Số người có tín hiệu: {meta.subject.nunique()}")
        md.append("- Bài tập: " + ", ".join(f"{k} ({v})" for k, v in meta.task.value_counts().items()))
        md.append("- Cổ tay: " + ", ".join(f"{k}={v}" for k, v in meta.wrist.value_counts().items()))
        per = meta.groupby("subject").size()
        md.append(f"- File/người: min {per.min()}, median {per.median():.0f}, max {per.max()}")
        # mẫu 30 file để đo tần số, số cột, độ dài, biên độ
        stats = []
        for p in meta.sample(min(30, len(meta)), random_state=0).path:
            df, sep = read_numeric(pathlib.Path(p))
            if df is None:
                continue
            arr = df.apply(pd.to_numeric, errors="coerce").to_numpy(float)
            fs = fs_from_time(arr[:, 0])
            stats.append(dict(file=pathlib.Path(p).name, n=len(arr), cols=arr.shape[1],
                              fs=None if fs is None else round(fs[0], 2),
                              t_unit=None if fs is None else fs[1],
                              jitter=None if fs is None else round(fs[2], 4),
                              acc_absmax=float(np.nanmax(np.abs(arr[:, 1:4]))) if arr.shape[1] >= 4 else None,
                              gyro_absmax=float(np.nanmax(np.abs(arr[:, 4:7]))) if arr.shape[1] >= 7 else None))
        if stats:
            sdf = pd.DataFrame(stats)
            md += ["", "Mẫu 30 file (cột 0 giả định là thời gian, 1–3 acc, 4–6 gyro — **cần anh xác nhận với readme của PADS**):",
                   "```", sdf.to_string(index=False), "```"]
            first = pathlib.Path(meta.path.iloc[0])
            md += ["", f"5 dòng đầu `{first.name}`:", "```", "".join(first.open().readlines()[:5]), "```"]
        meta.to_csv(REP / "pads_timeseries_index.csv", index=False)

    # observation JSON: mô tả bài tập
    obs = sorted(base.rglob("observation_*.json"))
    if obs:
        o = json.loads(obs[0].read_text())
        md += ["", f"## Observation JSON ({len(obs)} file) — ví dụ `{obs[0].name}`", "```json",
               json.dumps(o, indent=1, ensure_ascii=False)[:2500], "```"]
    readmes = [p for p in base.rglob("*") if p.name.lower().startswith("readme")]
    for r in readmes[:2]:
        md += ["", f"## `{r.relative_to(base)}` (2000 ký tự đầu)", "```", r.read_text(errors="ignore")[:2000], "```"]
    return md


# ---------------------------------------------------------------- Monipar
def audit_monipar():
    base = RAW / "monipar"
    md = ["# Audit Monipar", ""]
    if not base.exists():
        md.append("**Chưa có dữ liệu** — chạy scripts/01_download.sh"); return md
    rec = base / "_zenodo_record.json"
    if rec.exists():
        r = json.loads(rec.read_text())
        m = r.get("metadata", {})
        desc = re.sub(r"<[^>]+>", " ", html.unescape(m.get("description", "")))
        md += ["## Zenodo", f"- Tiêu đề: {m.get('title')}", f"- Ngày: {m.get('publication_date')}",
               f"- License: {m.get('license')}", f"- Phiên bản: {m.get('version')}",
               "- File: " + ", ".join(f"{f['key']} ({f.get('size',0)/1e6:.1f} MB)" for f in r.get("files", [])),
               "", "Mô tả:", "```", re.sub(r"\s+", " ", desc)[:3000], "```", ""]
    md.append("## Kho file")
    files = inventory(base, md)
    tab = [p for p in files if p.suffix.lower() in {".csv", ".txt", ".tsv", ".xlsx", ".xls", ".json", ".mat"}
           and p.name != "_zenodo_record.json"]
    md += ["", f"## Danh sách file dữ liệu ({len(tab)}), 80 file đầu", "```"]
    md += [str(p.relative_to(base)) for p in tab[:80]]
    md.append("```")
    # gợi ý tên bài tập từ tên file
    tokens = collections.Counter(t.lower() for p in tab for t in re.split(r"[_\-\s\.]+", p.stem) if t and not t.isdigit())
    md.append("- Token hay gặp trong tên file: " + ", ".join(f"{k}({v})" for k, v in tokens.most_common(40)))
    # mẫu nội dung
    md += ["", "## Mẫu nội dung (10 file)"]
    for p in tab[:10]:
        md.append(f"### `{p.relative_to(base)}`")
        try:
            if p.suffix.lower() in {".xlsx", ".xls"}:
                x = pd.read_excel(p, sheet_name=None, nrows=5)
                for s, d in x.items():
                    md += [f"sheet `{s}` shape≥{d.shape}", "```", d.head().to_string(), "```"]
                continue
            if p.suffix.lower() == ".json":
                md += ["```json", p.read_text(errors="ignore")[:1200], "```"]; continue
            if p.suffix.lower() == ".mat":
                md.append("(file .mat — cần scipy.io.loadmat, gửi tên biến cho Claude)"); continue
            df, sep = read_numeric(p)
            if df is None:
                md += ["```", "".join(p.open(errors="ignore").readlines()[:5]), "```"]; continue
            md.append(f"- sep `{sep}`, shape {df.shape}, cột: {list(df.columns)[:12]}")
            num = df.apply(pd.to_numeric, errors="coerce")
            for c in num.columns[:2]:
                fs = fs_from_time(num[c].to_numpy(float))
                if fs:
                    md.append(f"- cột `{c}` giống thời gian: fs ≈ {fs[0]:.2f} Hz (đơn vị {fs[1]}, jitter {fs[2]:.3f})")
            md += ["```", df.head(5).to_string(), "```"]
        except Exception as e:
            md.append(f"lỗi đọc: {e}")
    return md


if __name__ == "__main__":
    for name, fn in [("audit_pads.md", audit_pads), ("audit_monipar.md", audit_monipar)]:
        txt = "\n".join(fn()) + "\n"
        (REP / name).write_text(txt)
        print(f"== {name} ==\n{txt[:1500]}\n...")


In [ ]:
%%writefile scripts/00_env_report.py
"""Ghi cấu hình máy để Claude chọn batch size, framework, thời gian chạy."""
import importlib, os, platform, shutil, subprocess, pathlib

lines = ["# Môi trường", "", f"- OS: {platform.platform()}", f"- Python: {platform.python_version()}",
         f"- CPU cores: {os.cpu_count()}"]
try:
    mem = [l for l in open("/proc/meminfo") if l.startswith("MemTotal")][0].split()[1]
    lines.append(f"- RAM: {int(mem)/1e6:.1f} GB")
except Exception:
    pass
du = shutil.disk_usage(pathlib.Path(__file__).resolve().parents[1])
lines.append(f"- Ổ đĩa trống: {du.free/1e9:.0f} GB")
if shutil.which("nvidia-smi"):
    q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                       capture_output=True, text=True).stdout.strip()
    lines.append(f"- GPU: {q}")
else:
    lines.append("- GPU: không thấy nvidia-smi")
for m in ["numpy", "pandas", "scipy", "sklearn", "xgboost", "tensorflow", "torch"]:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "chưa cài"
    lines.append(f"- {m}: {v}")
out = pathlib.Path(__file__).resolve().parents[1] / "reports" / "env.md"
out.parent.mkdir(exist_ok=True)
out.write_text("\n".join(lines) + "\n")
print("\n".join(lines))


In [ ]:
!bash scripts/01_download.sh 2>&1 | tail -n 40

In [ ]:
!python scripts/02_audit.py | tail -n 60

In [ ]:
!python scripts/00_env_report.py
!du -sh data/raw/* reports